# QLoRA fine-tune: security log line → structured JSON

**Model:** `Qwen/Qwen2.5-1.5B-Instruct` (open weights, Apache-2.0)
**Method:** 4-bit QLoRA — base weights frozen, ~1% trainable adapter params
**Task:** parse a raw log line into a JSON event object
**Point of the project:** measure the same test set on the base model *and* the fine-tuned model, and report the gap.

Runtime → Change runtime type → **T4 GPU** before running anything.

## 1. Install

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate datasets
import torch
print(torch.__version__, torch.cuda.get_device_name(0))

## 2. Config

In [ ]:
BASE_MODEL   = "Qwen/Qwen2.5-1.5B-Instruct"
N_TRAIN      = 2000
N_VAL        = 200
N_TEST       = 300
SEED         = 42

LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

EPOCHS       = 2
LR           = 2e-4
BATCH_SIZE   = 4
GRAD_ACCUM   = 4
MAX_LEN      = 512
OUT_DIR      = "qwen1.5b-logparse-lora"

## 3. Synthetic dataset

Six log formats, each with its own field schema. The generator emits the log line *and* the
correct JSON at the same time, so the labels are exact by construction — no hand-annotation.

In [ ]:
import random, json, ipaddress

rng = random.Random(SEED)

MONTHS = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
USERS  = ["admin","root","ubuntu","svc_backup","jenkins","postgres","test","oracle",
          "s.parna","dev01","guest","ftpuser","nagios","www-data"]
HOSTS  = ["web-01","db-prod-2","bastion","edge-gw","app-07","mail-01"]
CMDS   = ["/usr/bin/apt update","/bin/cat /etc/shadow","/usr/sbin/service nginx restart",
          "/bin/rm -rf /var/log/auth.log","/usr/bin/systemctl stop firewalld","/bin/chmod 777 /opt"]
PATHS  = ["/index.php","/admin/login","/api/v1/users","/wp-admin/","/.env","/static/app.js",
          "/health","/api/v2/transactions"]
METHODS = ["GET","POST","PUT","DELETE","HEAD"]
STATUS  = [200,201,301,302,400,401,403,404,500,502]


def rand_ip():
    return str(ipaddress.IPv4Address(rng.randint(1 << 24, (1 << 32) - 1)))

def rand_priv_ip():
    return f"10.{rng.randint(0,255)}.{rng.randint(0,255)}.{rng.randint(1,254)}"

def ts_syslog():
    return f"{rng.choice(MONTHS)} {rng.randint(1,28):2d} {rng.randint(0,23):02d}:{rng.randint(0,59):02d}:{rng.randint(0,59):02d}"

def ts_clf():
    return f"{rng.randint(1,28):02d}/{rng.choice(MONTHS)}/2026:{rng.randint(0,23):02d}:{rng.randint(0,59):02d}:{rng.randint(0,59):02d} +0530"


def gen_ssh_fail():
    u, ip, port, host = rng.choice(USERS), rand_ip(), rng.randint(1024, 65535), rng.choice(HOSTS)
    invalid = rng.random() < 0.5
    line = (f"{ts_syslog()} {host} sshd[{rng.randint(1000,9999)}]: Failed password for "
            f"{'invalid user ' if invalid else ''}{u} from {ip} port {port} ssh2")
    return line, {"event": "failed_login", "service": "ssh", "user": u, "src_ip": ip, "port": port}


def gen_ssh_ok():
    u, ip, port, host = rng.choice(USERS), rand_ip(), rng.randint(1024, 65535), rng.choice(HOSTS)
    method = rng.choice(["password", "publickey"])
    line = (f"{ts_syslog()} {host} sshd[{rng.randint(1000,9999)}]: Accepted {method} for {u} "
            f"from {ip} port {port} ssh2")
    return line, {"event": "successful_login", "service": "ssh", "user": u, "src_ip": ip, "port": port}


def gen_sudo():
    u, cmd, host = rng.choice(USERS), rng.choice(CMDS), rng.choice(HOSTS)
    line = (f"{ts_syslog()} {host} sudo: {u} : TTY=pts/{rng.randint(0,4)} ; PWD=/home/{u} ; "
            f"USER=root ; COMMAND={cmd}")
    return line, {"event": "privilege_escalation", "service": "sudo", "user": u, "command": cmd}


def gen_nginx():
    ip, m, p, s = rand_ip(), rng.choice(METHODS), rng.choice(PATHS), rng.choice(STATUS)
    line = f'{ip} - - [{ts_clf()}] "{m} {p} HTTP/1.1" {s} {rng.randint(120, 90000)} "-" "Mozilla/5.0"'
    return line, {"event": "http_request", "service": "nginx", "src_ip": ip,
                  "method": m, "path": p, "status": s}


def gen_ufw():
    src, dst, dport, host = rand_ip(), rand_priv_ip(), rng.choice([22,23,80,443,3306,3389,8080,5432]), rng.choice(HOSTS)
    line = (f"{ts_syslog()} {host} kernel: [UFW BLOCK] IN=eth0 OUT= MAC=00:16:3e:aa:bb:cc "
            f"SRC={src} DST={dst} LEN={rng.randint(40,1500)} PROTO=TCP "
            f"SPT={rng.randint(1024,65535)} DPT={dport}")
    return line, {"event": "firewall_block", "service": "ufw", "src_ip": src,
                  "dst_ip": dst, "dst_port": dport}


def gen_windows():
    u, ip = rng.choice(USERS), rand_ip()
    lt = rng.choice([2, 3, 10])
    line = (f"EventID=4625 An account failed to log on. Subject: Security ID: NULL SID  "
            f"Account Name: {u}  Logon Type: {lt}  Source Network Address: {ip}  "
            f"Failure Reason: Unknown user name or bad password.")
    return line, {"event": "failed_login", "service": "windows", "user": u,
                  "src_ip": ip, "logon_type": lt}


GENERATORS = [gen_ssh_fail, gen_ssh_ok, gen_sudo, gen_nginx, gen_ufw, gen_windows]


def make_examples(n):
    seen, out = set(), []
    while len(out) < n:
        line, gold = rng.choice(GENERATORS)()
        if line in seen:
            continue
        seen.add(line)
        out.append({"log": line, "json": json.dumps(gold)})
    return out


data = make_examples(N_TRAIN + N_VAL + N_TEST)
train_data = data[:N_TRAIN]
val_data   = data[N_TRAIN:N_TRAIN + N_VAL]
test_data  = data[N_TRAIN + N_VAL:]

print(len(train_data), len(val_data), len(test_data))
for ex in train_data[:3]:
    print("\nLOG :", ex["log"])
    print("JSON:", ex["json"])

## 4. The prompt

The schema goes in the system prompt for **both** the baseline and the fine-tuned run. If you
only told the fine-tuned model what the fields are, the comparison would be rigged and the
result would mean nothing.

In [ ]:
SYSTEM = """You are a log parser. Convert the security log line into a single JSON object.

Use exactly one of these event schemas:
- failed_login (ssh): event, service, user, src_ip, port
- successful_login (ssh): event, service, user, src_ip, port
- privilege_escalation (sudo): event, service, user, command
- http_request (nginx): event, service, src_ip, method, path, status
- firewall_block (ufw): event, service, src_ip, dst_ip, dst_port
- failed_login (windows): event, service, user, src_ip, logon_type

Ports, status codes and logon types are integers. Output only the JSON object, nothing else."""


def to_messages(log, answer=None):
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": log}]
    if answer is not None:
        msgs.append({"role": "assistant", "content": answer})
    return msgs

## 5. Scoring

In [ ]:
def extract_json(text):
    """Pull the first balanced {...} block out of a model response."""
    text = text.replace("```json", "").replace("```", "").strip()
    start = text.find("{")
    if start == -1:
        return None
    depth = 0
    for i, ch in enumerate(text[start:], start):
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(text[start:i + 1])
                except json.JSONDecodeError:
                    return None
    return None


def norm(d):
    return {str(k): str(v) for k, v in d.items()}


def score(preds, golds):
    valid = exact = 0
    tp = fp = fn = 0
    for p_raw, g_raw in zip(preds, golds):
        g = norm(json.loads(g_raw))
        p = extract_json(p_raw)
        if p is None or not isinstance(p, dict):
            fn += len(g)
            continue
        valid += 1
        p = norm(p)
        if p == g:
            exact += 1
        for k, v in p.items():
            if k in g and g[k] == v:
                tp += 1
            else:
                fp += 1
        for k in g:
            if k not in p or p[k] != g[k]:
                fn += 1
    n = len(golds)
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec  = tp / (tp + fn) if tp + fn else 0.0
    f1   = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return {"valid_json": round(valid / n, 4),
            "exact_match": round(exact / n, 4),
            "field_f1": round(f1, 4)}

## 6. Load base model in 4-bit

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,   # T4 is Turing: fp16, not bf16
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    device_map={"": 0},
    attn_implementation="sdpa",             # flash-attn needs Ampere+, T4 can't use it
)
model.config.pad_token_id = tok.pad_token_id
print(model.get_memory_footprint() / 1e9, "GB")

## 7. Baseline — score the model *before* any training

In [ ]:
@torch.no_grad()
def generate(model, examples, batch_size=16, max_new_tokens=128):
    model.eval()
    tok.padding_side = "left"
    prompts = [tok.apply_chat_template(to_messages(ex["log"]), tokenize=False,
                                       add_generation_prompt=True) for ex in examples]
    outs = []
    for i in range(0, len(prompts), batch_size):
        chunk = prompts[i:i + batch_size]
        enc = tok(chunk, return_tensors="pt", padding=True, truncation=True,
                  max_length=MAX_LEN, add_special_tokens=False).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.pad_token_id)
        new = gen[:, enc["input_ids"].shape[1]:]
        outs.extend(tok.batch_decode(new, skip_special_tokens=True))
        print(f"  {min(i + batch_size, len(prompts))}/{len(prompts)}", end="\r")
    return outs


golds = [ex["json"] for ex in test_data]

base_preds = generate(model, test_data)
base_scores = score(base_preds, golds)
print("\nBASE MODEL:", base_scores)
print("\nsample output:\n", base_preds[0])

## 8. Attach LoRA adapters

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

## 9. Tokenize with prompt masking

Loss is computed on the JSON answer only. The prompt tokens get label `-100`, which tells
PyTorch to ignore them — otherwise the model wastes capacity learning to predict the schema
text you already hand it every time.

In [ ]:
from torch.utils.data import Dataset

class LogJsonDataset(Dataset):
    def __init__(self, examples):
        self.rows = []
        tok.padding_side = "right"
        for ex in examples:
            prompt = tok.apply_chat_template(to_messages(ex["log"]), tokenize=False,
                                             add_generation_prompt=True)
            full = prompt + ex["json"] + tok.eos_token
            p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
            f_ids = tok(full, add_special_tokens=False)["input_ids"][:MAX_LEN]
            labels = list(f_ids)
            for i in range(min(len(p_ids), len(labels))):
                labels[i] = -100
            self.rows.append({"input_ids": f_ids, "labels": labels})

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        return self.rows[i]


def collate(batch):
    maxlen = max(len(b["input_ids"]) for b in batch)
    input_ids, labels, mask = [], [], []
    for b in batch:
        pad = maxlen - len(b["input_ids"])
        input_ids.append(b["input_ids"] + [tok.pad_token_id] * pad)
        labels.append(b["labels"] + [-100] * pad)
        mask.append([1] * len(b["input_ids"]) + [0] * pad)
    return {"input_ids": torch.tensor(input_ids),
            "labels": torch.tensor(labels),
            "attention_mask": torch.tensor(mask)}


train_ds = LogJsonDataset(train_data)
val_ds   = LogJsonDataset(val_data)
print(len(train_ds), len(val_ds))

## 10. Train

In [ ]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=True,
    optim="paged_adamw_8bit",
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    gradient_checkpointing=True,
    report_to="none",
)

model.config.use_cache = False

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate,
)

trainer.train()
model.config.use_cache = True

## 11. Score the fine-tuned model on the same test set

In [ ]:
ft_preds = generate(model, test_data)
ft_scores = score(ft_preds, golds)
print("\nFINE-TUNED:", ft_scores)
print("\nsample output:\n", ft_preds[0])

In [ ]:
print(f"{'metric':<14}{'base':>10}{'tuned':>10}{'delta':>10}")
print("-" * 44)
for k in base_scores:
    b, f = base_scores[k], ft_scores[k]
    print(f"{k:<14}{b:>10.4f}{f:>10.4f}{f - b:>+10.4f}")

## 12. Look at what still fails

Do not skip this. The failure cases are the interesting part of the writeup — anyone can post
a metrics table, far fewer can say *which* inputs break the model and why.

In [ ]:
fails = [(ex["log"], ex["json"], p)
         for ex, p in zip(test_data, ft_preds)
         if extract_json(p) is None or norm(extract_json(p)) != norm(json.loads(ex["json"]))]

print(f"{len(fails)} / {len(test_data)} still wrong\n")
for log, gold, pred in fails[:10]:
    print("LOG :", log[:110])
    print("GOLD:", gold)
    print("PRED:", pred.strip()[:200])
    print("-" * 90)

## 13. Save the adapter

In [ ]:
model.save_pretrained(OUT_DIR)
tok.save_pretrained(OUT_DIR)
!du -sh {OUT_DIR}

# Optional: push to the Hub
# from huggingface_hub import notebook_login; notebook_login()
# model.push_to_hub("your-username/qwen1.5b-logparse-lora")

## 14. Reload the adapter later

The adapter is only a few tens of MB — the base model is fetched separately and stays untouched.

```python
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map={"": 0})
model = PeftModel.from_pretrained(base, OUT_DIR)
```